<!-- notebook-header -->
<a id="top"></a>
<a id="topic-graph-concepts"></a>

# 20. Graph Representation and Algorithm Selection

*LeetCode Playbook · topic notebook 20* · Graph Representation and Algorithm Selection.

← [Dynamic Programming](19_Dynamic_Programming.ipynb) · [Topic index](00_Topic_Index.ipynb#notebooks) · [A–Z problem finder](67_Problem_Finder.ipynb#finder) · [Breadth-First Search (BFS)](21_BFS.ipynb) →

Run the setup cell first, then the remaining cells from top to bottom. This notebook has its own imports and helpers.

In [1]:
# Setup: everything this notebook imports. Run this cell first.
from collections import Counter, defaultdict, deque, OrderedDict
from functools import cmp_to_key, lru_cache
from itertools import accumulate, combinations, permutations, product
import bisect, heapq, math, random, string

## Graph Representation and Algorithm Selection

> A graph is just *things* (nodes) and *you can get from this one to that one* (edges). BFS spreads from the start like a ripple in a pond, one ring of equal distance at a time; DFS runs down one corridor to its end before backing up. Both visit every node they can reach exactly once, because they mark a node the moment they discover it.

**Reach for it when** the problem talks about cells connected up, down, left and right, islands or regions, or something spreading in rounds: rot, fire, a signal. Reach for it too when it asks for the fewest steps, moves, jumps or transformations and every move costs the same, when it asks "can I reach", when a structure with cycles must be copied, or when a situation changes one move at a time: a board, a word, a lock.

**In this repo:** `graphs/` (19 of its 45 problems; Graphs II and III take the rest) · bank: `practice/simple/41_number_of_islands.py`, `practice/simple/42_clone_graph.py`, `practice/simple/44_rotting_oranges.py` · basics: `practice/simple/basics/graphs/01_adjacency_list_bfs_dfs.py`, `practice/simple/basics/searches/03_bfs_grid_shortest_path.py`, `practice/simple/basics/searches/04_dfs_recursive_and_iterative.py`, `practice/simple/basics/searches/05_connected_components.py`, `practice/simple/basics/searches/06_multi_source_bfs_01_matrix.py`

### The picture

Half of every graph problem is one sentence: *what is a node, and what is an edge?* Once you can say it, the code is one of two loops.

In a grid of land and water, a node is a cell `(r, c)` and an edge is a step up, down, left or right onto land, so "how many groups?" is a flood fill, by DFS or by BFS. When rot spreads every minute, an edge is one minute of spreading, and "when is the last cell reached?" is a BFS from every rotten cell at once.

The node is whatever changes from one move to the next. When a word changes one letter at a time, the node is a word, joined to every word one letter away. When tiles slide into a gap, it is the whole board as a string; when keys open locks, a cell with the keys in hand; when the question counts buses, a bus route, joined to every route that shares a stop. All of them ask for the fewest moves, so all of them are BFS.

A graph you must deep-copy is walked once, with a map from every old node to its copy. Weighted edges are [Graphs III](00_Topic_Index.ipynb#s19); "in what order?" and "are these connected?" are [Graphs II](00_Topic_Index.ipynb#s18).

```text
A grid is a graph you never build:              BFS from S spreads in rings of equal distance:

              (r-1, c)                          S . . #            0 1 2 #
    (r, c-1)   (r, c)   (r, c+1)                . # . .    -->     1 # 3 4
              (r+1, c)                          . . . #            2 3 4 #

each cell's edges go to the 4 cells             queue over time:  [0] [1 1] [2 2] [3 3] [4 4]
around it that are on the board                 ring k leaves the queue before ring k+1 starts

DFS on the same grid runs down one corridor as far as it can, then backs up to the last fork.
Same cells, same cost, but its order says nothing about distance.
```

**Why it is fast.** The brute force in most of these problems starts a fresh search from every cell. On a grid of one big island, every search re-walks the whole island, so the work is O((rows · cols)²). One `seen` set shared by all the searches is the whole speed-up.

With a shared `seen`, every node goes into the container at most once, because it is marked the moment it goes in, and every edge is looked at once from each end. A traversal therefore costs O(V + E), which on a grid is O(rows · cols): each cell is pushed once and looks at its four neighbours once.

### From idea to code

**The idea in one sentence:** *put the start in a container and mark it; then repeatedly take one node out, and put in every neighbour you have never seen, marking it as it goes in.*

A queue, oldest out first, makes that BFS. A stack, newest out first, makes it DFS, and that is all a flood fill needs. Only a few jobs need the real DFS tree, with its "on the current path" information: cycle colours, topological order and bridges, all in [Graphs II](00_Topic_Index.ipynb#s18). Those use recursion.

The seven decisions for the fewest-steps BFS read as one story. The **state** is a `deque` frontier and a `dist` map, and `dist` doubles as `seen`. By **definition**, `dist[v]` is the fewest moves from the start to `v`, and the queue holds the discovered cells not expanded yet. The **invariant**, true at the end of every step, is that the queue holds at most two distances, the `d`s in front of the `d + 1`s, and that every cell in `dist` is waiting in the queue or already expanded, so it entered exactly once.

A **step** pops a cell and gives every neighbour that is on the board, passable and new the distance `d + 1` as it joins the queue. That is the **record** too: the distance is written at discovery, and it is already final. **Init** puts every source in the queue and in `dist` at distance 0, and the **return** is the target's distance when it is *popped*, or -1 if the queue runs dry.

For counting and measuring regions the story changes little. The state is a `seen` set of the cells already pushed, a stack for the current flood, and two counters: `count`, the floods started so far, and `best`, the largest area. The invariant is that every cell in `seen` is waiting in the stack or already expanded, so it enters the stack exactly once, and that a finished flood has put its whole region into `seen`.

A step pops a cell, then marks and pushes every new land neighbour. The outer loop records `count += 1` when it finds unseen land, and the flood records `best = max(best, area)` when it ends. Everything starts empty with `count = best = 0`, and the return is `count` and `best`.

The sentences you say map onto the lines you type. "The four cells around me" is `for dr, dc in DIRS: nr, nc = r + dr, c + dc`, and "still on the board" is `0 <= nr < rows and 0 <= nc < cols`. "Never discovered" is `(nr, nc) not in seen`, or `not in dist`.

"Mark it as I discover it" is `seen.add(nxt)` on the line just before `queue.append(nxt)`. "The oldest discovery next" is `queue.popleft()`, which makes it BFS; "the newest discovery next" is `stack.pop()`, which makes it DFS. "One ring further out" is `dist[nxt] = dist[cur] + 1`, and "a whole ring at once" is `for _ in range(len(queue)):`. "A new island starts here" is an outer loop over every cell with `if land and not seen: count += 1`.

Turn the story into a graph before you type. Six questions decide every line of a BFS:

1. **Node**: what changes from one move to the next? A cell `(r, c)`, a word, a board string or `(r, c, keys)`; whatever it is goes into `seen`, so it must be hashable, a tuple or a string.
2. **Edge**: what is one legal move? Step onto a free cell, change one letter, turn one wheel; it becomes `def neighbours(node): yield ...`.
3. **Cost**: does every move cost the same? Yes means BFS; only 0 or 1 means a deque, the 0-1 BFS; any w ≥ 0 means a heap, in [Graphs III](00_Topic_Index.ipynb#s19).
4. **Start**: one source, or many at once? The entrance, or every rotten orange, or every 0; all of them go into the queue and into `seen` at distance 0.
5. **Goal**: what ends the search? Reaching a cell, holding every key, or running to the end; `if is_goal(node): return d`, checked when the node is *popped*.
6. **Extra**: can two arrivals at the same place have different futures? More keys, more budget left, a different set of visited nodes; then the extra goes into the node, and places × extras must fit in memory, up to about 10⁶ states.

Never put the move count itself in the node: the queue order already carries it, and with it inside, `seen` can no longer merge two arrivals at the same place. On an open 6 × 6 grid, a BFS from corner to corner pops 36 states when the node is the cell, and 111 when it is `(r, c, moves)`.

Step zero is listing a node's neighbours. From an edge list you build an adjacency list once: `graph[u]` holds every node one edge away from `u`, and an undirected edge is written into both lists. For the six nodes and five edges below, node 3 ends up next to 1, 2 and 4.

On a grid you store no edges at all. The neighbours of `(r, c)` are the four cells around it that are still on the board, computed on the fly, and the corner (0, 0) of a 3 × 3 grid has only two of them.

In [2]:
def build_graph(n, edges, directed=False):
    graph = [[] for _ in range(n)]               # graph[u] = the nodes one edge away from u
    for u, v in edges:
        graph[u].append(v)
        if not directed:
            graph[v].append(u)                   # undirected: store both directions
    return graph


DIRS = ((1, 0), (-1, 0), (0, 1), (0, -1))        # down, up, right, left


def grid_neighbours(grid, r, c):
    """The in-bounds cells next to (r, c). A grid is a graph whose edges you never store."""
    for dr, dc in DIRS:
        nr, nc = r + dr, c + dc
        if 0 <= nr < len(grid) and 0 <= nc < len(grid[0]):
            yield nr, nc


print(build_graph(6, [(0, 1), (0, 2), (1, 3), (2, 3), (3, 4)]))   # [[1, 2], [0, 3], [0, 3], [1, 2, 4], [3], []]
print(build_graph(3, [(0, 1), (1, 2)], directed=True))            # [[1], [2], []]
print(list(grid_neighbours(["...", "...", "..."], 0, 0)))         # [(1, 0), (0, 1)]  a corner has 2
print(list(grid_neighbours(["...", "...", "..."], 1, 1)))         # [(2, 1), (0, 1), (1, 2), (1, 0)]

[[1, 2], [0, 3], [0, 3], [1, 2, 4], [3], []]
[[1], [2], []]
[(1, 0), (0, 1)]
[(2, 1), (0, 1), (1, 2), (1, 0)]


**Try it**
- Compare `build_graph(2, [(0, 1)], directed=True)` with `build_graph(2, [(0, 1)])`: `[[1], []]` versus `[[1], [0]]`. In the directed one, node 1 has no way back to 0.
- Feed 1-indexed edges: `build_graph(3, [(1, 2), (2, 3)])` raises `IndexError` (there is no `graph[3]`). When nodes are numbered from 1, allocate `n + 1` lists.
- An adjacency matrix becomes lists in one line: for `M = [[0, 1, 1], [1, 0, 0], [1, 0, 0]]`, `[[j for j, x in enumerate(row) if x] for row in M]` gives `[[1, 2], [0], [0]]`.

### Where it goes wrong

1. **Marking `seen` when you pop instead of when you push.** Between its first push and its pop, a node is pushed again by every other neighbour that reaches it, so the queue fills with copies, and anything counted per pop is counted more than once: a 2×2 block of land gets area 5. The other correct form is to mark on pop *and skip a node that is already marked*: copies still enter the queue, but none is expanded twice. Dijkstra must use that form ([Graphs III](00_Topic_Index.ipynb#s19)).
2. **Half a bounds check.** Without `0 <=`, Python reads `grid[-1]` as the last row instead of failing, so the top edge silently touches the bottom edge: `islands(["1", "0", "1"])` reports `(2, 2)` instead of `(2, 1)`. Always write `0 <= nr < rows`.
3. **Forgetting to mark the start.** Without `seen.add((r, c))` before the flood, the start is pushed again by its own neighbour: `islands(["11"])` returns `(1, 3)`.
4. **One search on a disconnected graph.** With `build_graph(4, [(0, 1), (2, 3)])`, a search from 0 reaches only {0, 1}. Counting or visiting everything needs the outer loop that tries every node as a start.
5. **DFS for "fewest steps".** DFS finds *a* path, not the shortest: on an open 3×4 grid a stack-based search reports 8 moves from (0, 0) to (2, 0), BFS reports 2.
6. **Counting pops instead of rings.** When the answer is a number of rounds, drop the ring loop (or put `minutes += 1` inside it) and you count pops: the first Rotting Oranges grid answers 6 instead of 4. Finish the whole ring, then add one.
7. **Deep recursion.** A recursive flood of a 300×300 island can need 90,000 nested calls; Python stops at about 1,000. Use an explicit stack (or BFS) on big grids, and say so.
8. **Flood fill with the new colour equal to the old one.** A fill that uses the paint as its "seen" mark never ends on `[[1, 1]]` from (0, 0) with colour 1: painting changes nothing, so the two cells keep pushing each other. Return early when the colours are equal.
9. **An undirected edge stored once.** `build_graph(2, [(0, 1)], directed=True)[1]` is `[]`: from node 1 there is no way back.
10. **Positions given as lists.** LeetCode passes `[0, 0]`: `shortest_steps(maze, [0, 0], (1, 3))` without the conversion raises `TypeError: unhashable type: 'list'`, and a list `target` silently returns -1 because `(1, 3) == [1, 3]` is False. Convert first: `start, target = tuple(start), tuple(target)`.
11. **`1` versus `"1"`.** Number of Islands stores the strings `"1"`/`"0"`; Max Area of Island and Number of Enclaves, which counts the land cells that cannot walk off the grid, store the ints. `islands([[1, 1], [0, 1]])` returns `(0, 0)`, because `1 != "1"`.
12. **`seen` keyed on less than the whole state.** In BFS over states, `(cell, keys)` is the node, not `cell`. Shortest Path to Get All Keys asks for the fewest moves to collect every key when each lock needs its key; with `seen` keyed on the cell, `["a.@A.b"]` answers -1 instead of 7, because the walk must come back over cells it has already crossed.

### Edge cases to say out loud

Start equals target · start or target is a wall · target unreachable · no land at all · one big island · diagonal cells (not neighbours unless the problem says 8 directions) · nodes with no edges · nodes numbered from 1 · positions given as lists.

### Variations

Every variation below keeps the loop and changes one thing: where the search starts, what a node is, or what rides along with it. The table is the overview; each variation then gets its own paragraph and, for the important ones, its code.

| Variation | What changes from the template | Problems |
|---|---|---|
| **Count / measure regions** | outer loop over every cell; count the launches; the flood returns its size | Number of Islands (200), Max Area of Island (695), Flood Fill (733) |
| **Multi-source BFS** | push *every* source before the loop; a ring = one minute, or one step from the nearest source | Rotting Oranges (994), Walls and Gates (286), 01 Matrix (542) |
| **Return the path** | a `parent` map instead of `dist`; walk it back from the target | any shortest path |
| **8 directions, blocked start** | 8 entries in the direction list; check the start cell before the loop | Shortest Path in Binary Matrix (1091) |
| **Border-first** | flood from the border instead of asking every cell; what stays unmarked is enclosed | Surrounded Regions (130), Number of Enclaves (1020), Pacific Atlantic Water Flow (417) |
| **Two floods** | flood one island to collect it, then multi-source BFS from all its cells toward the other | Shortest Bridge (934) |
| **Copy a graph** | an `old -> new` dict is both `seen` and the lookup for neighbours | Clone Graph (133) |
| **Carry a label across edges** | a colour flips (bipartite) or a ratio multiplies (division) along each edge | Is Graph Bipartite (785), Possible Bipartition (886), Evaluate Division (399) |
| **BFS over states** | the node is a string or a tuple; neighbours come from a function; `bfs_states` is unchanged | Open the Lock (752), Word Ladder (127) |
| **BFS over augmented states** | node = (cell, extra); `seen` keys on the whole tuple | Shortest Path in a Grid with Obstacles Elimination (1293) |
| *Second pass:* **richer nodes** | the node is a whole board, (cell, keys held) or (node, visited mask) | Sliding Puzzle (773), Shortest Path to Get All Keys (864), Shortest Path Visiting All Nodes (847) |
| *Second pass:* **BFS over groups** | expand a whole group (a bus route, all equal values) once, then never again | Bus Routes (815), Jump Game IV (1345) |
| *Second pass:* **pruned branching** | branch only on moves that fix the first mismatch | K-Similar Strings (854) |
| *Second pass:* **every shortest path** | keep every parent from the previous ring; walk back from the end | Word Ladder II (126) |

Flood Fill is the smallest member of the first row: recolour the start pixel and every pixel of the same colour connected to it, so `[[1, 1, 1], [1, 1, 0], [1, 0, 1]]` painted with 2 from the middle becomes `[[2, 2, 2], [2, 2, 0], [2, 0, 1]]`. The paint itself serves as the visited mark, which is why trap 8 exists.

Shortest Path in Binary Matrix is the template with 8 directions and a start cell that may itself be blocked: the fewest cells on a path of 0s from the top-left to the bottom-right corner, moving in any of 8 directions, or -1.

The multi-source variation comes first because it changes only the first line of the template. Rotting Oranges is worked line by line in [From Idea to Code](02_Idea_to_Code.ipynb#topic-idea-to-code). The same idea with distances instead of minutes is 01 Matrix: for every cell, how far is the nearest 0, so `[[0, 1, 1, 1, 0]]` becomes `[[0, 1, 2, 1, 0]]`.

Every 0 is a source, so all of them go into the queue at distance 0 before the loop starts, and the first time the wave reaches a cell it came from the *nearest* 0. Walls and Gates asks for the distance from every empty room to its nearest gate, with walls in the way; it is the same code with the gates as the zeros, and a room the wave never reaches keeps its starting value.

Jump Game IV is the same idea on an array. From index i you may jump to i − 1, i + 1 or any index holding the same value, and it asks for the fewest jumps from the first index to the last: 3 for `[100, -23, -23, 404, 100, 23, 23, 23, 3, 404]`. Run BFS over indexes and delete a value's bucket of equal indexes the first time it is used, so a long run of equal values is read once instead of on every pop.

K-Similar Strings asks for the fewest swaps of two letters that turn one anagram into another: `"abac"` becomes `"baca"` in 2. It is BFS over strings that branches only on swaps putting the right letter into the first wrong slot, and some shortest sequence always begins with such a swap, so the pruning loses nothing.

Word Ladder II asks for *every* shortest ladder, not just its length, so the ring structure has to be kept. Run the BFS ring by ring and record, for every word, *all* the words of the previous ring that reach it, a parents map.

Remove a ring's words from the dictionary only after the whole ring is done, so that two parents in the same ring can both register. Stop after the ring that contains `endWord`, then walk the parents map backwards from `endWord` to list every path.

### Say it in the interview

Regions:

> "Each land cell is a node joined to its land neighbours, so an island is a connected component. A fresh search from every cell re-walks each island, O((mn)²); with one shared `seen` every cell is pushed once, O(mn). Unseen land launches a new flood: one more island. I flood with an explicit stack so a big island can't hit the recursion limit."

Fewest moves:

> "Every move costs 1, so BFS. A cell at distance k + 1 can only be discovered from a cell at distance k, and ring k leaves the queue before ring k + 1, so the first time the target is popped its distance is minimal. I mark on enqueue, so each cell enters once: O(mn)."

Point at the `seen.add` right before the `append`, the bounds check with both halves, and the line where you record. For a state search, answer the six questions out loud before writing any code.

Expect follow-ups, and have the one-line answer ready. The path itself is a parent map. Eight directions is a longer direction list. "Don't modify the input" is a `seen` set instead of sinking cells. Recursion depth is an explicit stack. A Word Ladder that is too slow is bidirectional BFS: search from both ends and always expand the smaller frontier.

### Problem map

| Problem | Where | Key insight |
|---|---|---|
| 01 Matrix | `graphs/zero_one_matrix.py` | multi-source BFS from every 0; a cell's first visit is its distance |
| Bus Routes | `graphs/bus_routes.py` | BFS over routes, not stops: a stop → routes index; board each route once, expand each stop once |
| Clone Graph | `graphs/clone_graph.py` · `practice/simple/42_clone_graph.py` | one traversal; the old → new dict is both the visited set and the neighbour lookup |
| Flood Fill | `graphs/flood_fill.py` | recolouring is the visited mark; return early if the new colour equals the old one |
| Jump Game IV | `graphs/jump_game_iv.py` | BFS on indexes; a value's bucket of equal indexes is used once, then deleted |
| K-Similar Strings | `graphs/k_similar_strings.py` | BFS over strings; branch only on swaps that fix the first mismatched slot |
| Max Area of Island | `graphs/max_area_of_island.py` | flood each island once from its first cell; the flood returns its size |
| Number of Enclaves | `graphs/number_of_enclaves.py` | flood the land the border can reach; the land left over is enclosed |
| Number of Islands | `graphs/number_of_islands.py` · `practice/simple/41_number_of_islands.py` | count how many times the scan has to start a new flood |
| Pacific Atlantic Water Flow | `graphs/pacific_atlantic_water_flow.py` | climb uphill from each ocean's border; answer = cells both floods reach |
| Rotting Oranges | `graphs/rotting_oranges.py` · `practice/simple/44_rotting_oranges.py` | multi-source BFS from all rotten oranges; one ring = one minute; -1 if fresh ones remain |
| Shortest Path in a Grid with Obstacles Elimination | `graphs/shortest_path_in_a_grid_with_obstacles_elimination.py` | state = (row, col, eliminations left); more budget at the same cell is a different state |
| Shortest Path to Get All Keys | `graphs/shortest_path_to_get_all_keys.py` | state = (row, col, key bitmask); the goal is any cell with the full mask |
| Shortest Path Visiting All Nodes | `graphs/shortest_path_visiting_all_nodes.py` | state = (node, visited mask); seed all n starts; the first full mask wins |
| Sliding Puzzle | `graphs/sliding_puzzle.py` | the board as a 6-char string; neighbours = swap the blank with a touching index |
| Surrounded Regions | `graphs/surrounded_regions.py` | mark the O's reachable from the border as safe; flip every other O |
| Walls and Gates | `graphs/walls_and_gates.py` | multi-source BFS from all gates; the first write into a room is final, so INF itself means "unvisited" |
| Word Ladder | `graphs/word_ladder.py` | BFS on words; wildcard buckets ("h*t") list neighbours without comparing all pairs |
| Word Ladder II | `graphs/word_ladder_ii.py` | BFS by rings keeping every parent from the previous ring; backtrack from endWord |

### Self-check

1. Why mark a node as seen when you push it, not when you pop it?
<details><summary>Answer</summary>Between being pushed and being popped, a node can be discovered again by other neighbours. If it is only marked at pop time, every discovery pushes another copy: the queue grows, work repeats, and anything counted per pop (like an area) is counted more than once. Marking on push lets each node into the container exactly once. (Marking on pop also works if you skip nodes that are already marked; that is the form Dijkstra needs.)</details>

2. Why does BFS give the fewest moves while DFS does not?
<details><summary>Answer</summary>The queue only ever holds distances <code>d</code> and <code>d + 1</code>, in that order, so nodes come out in order of distance; when the target first comes out, no shorter path can still be waiting. DFS follows whichever corridor it tried first, so it reaches the target along <em>some</em> path, not necessarily a short one.</details>

3. In Shortest Path to Get All Keys, why can't <code>seen</code> hold just the cell?
<details><summary>Answer</summary>The same cell with more keys is a different situation: locks that were walls are now open. A walk often has to come back over cells it has already crossed, for example to fetch a key behind it. With cell-only <code>seen</code>, <code>["a.@A.b"]</code> answers -1 instead of 7.</details>

4. Why should the move count never be part of the node?
<details><summary>Answer</summary>The queue order already tells you the distance (or the distance rides along beside the state). With the count inside, the same cell reached after 3 moves and after 5 moves are two different nodes, so <code>seen</code> cannot merge them and the search re-explores places it has already been: on an open 6 × 6 grid, 111 pops instead of 36.</details>

## Graphs II: Ordering & Connectivity

> Two questions about how nodes relate. **Order**: *what can I do first?* Keep taking a node that nothing is waiting on (topological sort). **Connectivity**: *are these two in the same group?* Every group has a boss; merging two groups means pointing one boss at the other (union-find).

[Graphs I](00_Topic_Index.ipynb#s17) walked a graph to measure distances and regions. This section asks two other questions, in what order the nodes can be taken and which nodes belong together, and answers each with a small structure instead of a fresh search.

**Reach for it when** the problem talks about order: prerequisites, dependencies, "must come before", a build order, an alphabet hidden in sorted words, or the minimum time when tasks run in parallel. Reach for it too when it talks about groups: merge, same group, connected, edges that arrive one at a time, "the extra edge that makes a cycle", "after each operation, how many groups?", or queries with a weight limit.

"Which cable, if cut, disconnects the network?" is bridges, and "use every ticket exactly once" is an Eulerian path; both come in the second pass at the end of Variations.

**In this repo:** `graphs/` (19 of its 45 problems) · bank: `practice/simple/43_course_schedule.py`, `practice/simple/47_redundant_connection.py` · basics: `practice/simple/basics/graphs/02_topological_sort_kahn_and_dfs.py`, `practice/simple/basics/graphs/03_union_find.py`

### The picture

```text
TOPOLOGICAL SORT (Kahn): keep taking whatever nothing is waiting on

      0 -----> 1            indegree = how many arrows point INTO a node
      |        |
      v        v            start      indegree  0:0  1:1  2:1  3:2    ready [0]
      2 -----> 3            take 0  ->           1:0  2:0  3:2         ready [1, 2]
                            take 1  ->                     3:1         ready [2]
                            take 2  ->                     3:0         ready [3]
                            take 3  ->  order 0 1 2 3

A cycle 1 -> 2 -> 3 -> 1: each waits on another, none ever reaches 0, and "ready" runs dry early.
```

```text
UNION-FIND: every group is a tree, and its root is the group's name

  union(0, 1)   union(2, 3)          union(1, 3): find(1) = 0, find(3) = 2,
                                     so hang root 2 under root 0
     0     2                              0
     |     |                             / \
     1     3                            1   2
                                            |
                                            3      find(3) walks 3 -> 2 -> 0; path halving then
                                                   points 3 at its grandparent, the root
  union(0, 2) now: find(0) == find(2) == 0, so it returns False: they were already connected.
```

Kahn's algorithm gives an order and a cycle check in one pass: it is iterative, so it never hits the recursion limit, and a cycle shows up as `len(order) < n`. DFS colours are the choice when you are already inside a DFS, or when the question is which nodes can reach a cycle: Find Eventual Safe States (802) asks for the nodes from which every path ends at a node with no way out, which are exactly the nodes that cannot reach a cycle.

For connectivity, a graph given once needs nothing more than one BFS or DFS ([Graphs I](00_Topic_Index.ipynb#s17)). Union-find earns its place when edges arrive over time, when many "connected?" questions come between the arrivals, or when the question is the edge that closes a cycle.

**Why it is fast.** The brute-force ordering rescans every edge each round to find a node that is ready: O(V · (V + E)). Kahn keeps one counter per node, and taking a node only changes the counters of *its* successors, so each edge is touched once: O(V + E).

For connectivity, re-running a BFS for every "connected?" question costs O(V + E) each time. Groups only ever merge, so a forest of parent pointers can answer in nearly O(1), as long as the trees stay flat.

Union by size keeps them flat: a node goes one level deeper only when its group is hung under a group at least as big, so its group at least doubles each time, and no tree grows deeper than log₂ n. Path halving flattens the rest. Together they cost α(n) per operation amortised, that is, averaged over any long run of operations; α is the inverse Ackermann function, which stays below 5 for any n you will ever store.

### From idea to code

**The idea in two sentences:** *Order: count each node's unfinished predecessors, take any node whose count is 0, and taking it lowers its successors' counts.* *Connectivity: `find` walks up to the root that names a group, `union` points one root at the other, and a `union` whose two roots are already equal has just found an edge that closes a cycle.*

For Kahn's algorithm the **state** is three things: the graph, where `graph[u]` lists every v that u must precede, an `indegree` list, and a queue of ready nodes. The **definition**: `indegree[v]` counts the predecessors of v not taken yet, where a node's indegree is the number of arrows pointing into it. The **invariant**: every node in the queue has all its predecessors already in `order`. A **step** pops u, appends it to `order`, and lowers `indegree[v]` for each successor v, pushing v the moment its count reaches 0.

The **record** is the pop order itself, which is a valid order. **Init** puts *every* node of indegree 0 in the queue, isolated nodes too. The **return** is `order` when `len(order) == n`; anything shorter means a cycle, translated into `[]`, `False` or `""`.

Union-find makes the same decisions about groups. Its state is `parent[x]`, `size[root]` and, when asked, the number of groups. By definition `find(x)` is the root of x's tree, and two nodes share a group exactly when they share a root. The invariant: every group is one tree, and `r` is a root exactly when `parent[r] == r`. A step, `union(a, b)`, finds both roots and, if they differ, hangs the smaller tree under the bigger.

`union` also records: it returns `False` when the two were already connected, so the edge closes a cycle, and `True` when there is one group fewer. Init is `parent = list(range(n))`, `size = [1] * n` and `count = n`, and the return is the count, the redundant edge, or the groups bucketed by `find`.

Course Schedule asks whether n courses can all be taken when a pair `[a, b]` means "take b before a"; Course Schedule II asks for one such order, or `[]` when none exists. With 4 courses and `[[1, 0], [2, 0], [3, 1], [3, 2]]`, one order is `[0, 1, 2, 3]`. The template keeps one convention, an edge (u, v) for "u before v", so the wrapper converts the pairs once and never thinks about them again. In the code, "everything I can do right now" is the starting queue, and "u is done, so v waits on one fewer" is `indegree[v] -= 1`.

### Where it goes wrong

1. **Edge direction.** `[a, b]` means "b before a", so the edge is `b -> a`. Reversed edges give the order backwards (`[3, 1, 2, 0]` above), and Course Schedule I still passes because reversing edges keeps cycles, so the bug hides until part II. Convert once, at the top.
2. **Forgotten nodes.** Seed the queue with *every* node of indegree 0, including nodes with no edges at all. Alien Dictionary, which recovers an alphabet from words sorted in it, makes every letter a node, even one that appears in no rule: `["ab", "adc"]` must output `c` too.
3. **Duplicate edges with a set adjacency.** If `graph[x]` is a set, add to `indegree[y]` only when `y` was really new. Without that guard, `alien_order(["ab", "ac", "bb", "bc"])` returns `""` instead of `"abc"`: the fact "b before c" was counted twice but can only be freed once. With list adjacency duplicates are harmless: counted twice, freed twice.
4. **Two colours in DFS.** "Visited" is not "on my path": with two colours, the diamond `0->1, 0->2, 2->1` looks like a cycle.
5. **Linking nodes instead of roots.** `parent[b] = a` can pull `b` out of its old group (`union(0, 1)`, then `union(2, 1)` leaves 0 and 1 apart). Always link `find(b)` under `find(a)`.
6. **No halving, no union by size.** With a plain `parent[find(a)] = find(b)`, the unions `union(0, k)` for k = 1 … 999 build one long chain, and `find(0)` then walks 999 links.
7. **Counting every union.** Lower the group count only when the roots differed; a repeated edge must not change it. Four nodes with the edges `[0, 1]`, `[1, 0]` and `[2, 3]` form 2 groups, but counting every edge, `n - len(edges)`, says 1.
8. **1-indexed nodes.** Courses, people and tree nodes are often numbered `1..n`: `DSU(3).union(1, 3)` raises `IndexError`. Allocate `n + 1`.
9. **Comparing parents instead of roots.** After `union(0, 1)`, `union(2, 3)`, `union(0, 2)` the parents are `[0, 0, 0, 2]`: `parent[3] == parent[1]` is False although `find(3) == find(1)`. Only roots name groups.

### Edge cases to say out loud

One node · no edges (any order is valid, every node is its own group) · a self-loop (a cycle of length 1; `union(x, x)` is False) · duplicate edges · several separate chains · a word that comes before its own prefix (Alien Dictionary) · one node and no edges is a valid tree.

### Variations

Every variation keeps one of the two templates and changes what goes in or what is read out: which nodes start, what an edge means, or what is done with the groups at the end.

| Variation | What changes from the template | Problems |
|---|---|---|
| **Is any order possible?** | count the pops; possible iff all n come out | Course Schedule (207) |
| **Return an order** | keep the pop order | Course Schedule II (210) |
| **Smallest or unique order** | a heap instead of the deque; the order is unique iff the queue never holds two nodes | Sequence Reconstruction (444): is one given sequence the only order the rules allow |
| **Derive the edges first** | compare neighbouring words; their first different letter is one edge; prefix check | Alien Dictionary (269) |
| **Longest path in a DAG** | in topological order, push `finish[v] = max(finish[v], finish[u] + time[v])` | Parallel Courses III (2050) |
| **Peel leaves** | Kahn on an undirected tree: remove degree-1 leaves ring by ring; the last 1–2 nodes are the centre | Minimum Height Trees (310) |
| **Tree check / component count / judge** | n − 1 edges and no failed union; `dsu.count`; in-degree minus out-degree | Graph Valid Tree (261), Number of Connected Components (323), Find the Town Judge (997) |
| **The cycle edge, undirected** | the first union that returns False | Redundant Connection (684) |
| **Group by a shared key** | each key (an email) remembers its first owner; union with it | Accounts Merge (721) |
| **Union, then act per component** | sort letters inside each component; stones − groups | Smallest String With Swaps (1202), Most Stones Removed (947) |
| **Equations as unions** | union every `a == b`, then check that no `a != b` falls inside one group | Satisfiability of Equality Equations (990): can every `==` and `!=` hold at once |
| **Two-colouring with a DSU** | every neighbour of u joins one group, and that group must not contain u | Is Graph Bipartite (785): two camps, every edge between them, as in [Graphs I](00_Topic_Index.ipynb#s17) |
| **Online counting** | a new node adds 1; each successful union subtracts 1 | Number of Islands II (305) |
| *Second pass:* **bridges** | DFS low-link: tree edge (u, v) is a bridge iff `low[v] > disc[u]` | Critical Connections in a Network (1192) |
| *Second pass:* **use every edge once** | Hierholzer: write a node down when it is stuck, reverse at the end | Reconstruct Itinerary (332) |
| *Second pass:* **prime hubs** | each prime factor remembers its first owner; union with it | Largest Component Size by Common Factor (952) |
| *Second pass:* **unions over time** | one timestamp at a time; reset everyone not connected to person 0 | Find All People With Secret (2092) |
| *Second pass:* **offline thresholds** | sort queries by limit, let edges in by weight, answer into the original slots | Checking Existence of Edge Length Limited Paths (1697) |
| *Second pass:* **an order inside an order** | topo-sort the items and the groups separately; output groups in order, items inside | Sort Items by Groups Respecting Dependencies (1203) |
| *Second pass:* **the cycle edge, directed** | a node with two parents first: try dropping the later edge, else the earlier one | Redundant Connection II (685) |
| *Second pass:* **two players** | two DSUs; shared edges first, then each player's own | Remove Max Number of Edges to Keep Graph Fully Traversable (1579) |
| *Second pass:* **facts per component** | per root: its size and how many infected nodes it holds | Minimize Malware Spread (924) |

Three warm-ups need almost nothing beyond the templates. Graph Valid Tree asks whether n nodes and a list of undirected edges form one tree, and `5, [[0, 1], [0, 2], [0, 3], [1, 4]]` does: a tree on n nodes has exactly n − 1 edges, none of them wasted on a loop. Number of Connected Components counts the groups, 2 for `5, [[0, 1], [1, 2], [3, 4]]`: start at n, and every real merge removes one.

Find the Town Judge looks for the one person who is trusted by everybody else and trusts nobody, or -1: with 3 people and `[[1, 3], [2, 3]]`, the judge is 3. It needs no graph at all, only a score per person, in-degree minus out-degree, and only the judge reaches n − 1.

Seven more Hard problems need no new code, only one more idea each:

- Largest Component Size by Common Factor (952) joins two numbers when they share a factor greater than 1 and asks for the size of the biggest group, 4 for `[4, 6, 15, 35]`. It is Accounts Merge with primes as the emails: union each number with the first owner of each of its prime factors, then count the numbers per root.
- Find All People With Secret (2092): person 0 tells `firstPerson` a secret at time 0, and a meeting `[x, y, t]` passes it between x and y, instantly along chains of meetings at the same time; who knows it in the end? Union-find can merge but never split, so union one timestamp's meetings at a time, then reset everyone from that timestamp who is not connected to person 0.
- Checking Existence of Edge Length Limited Paths (1697) asks, for each query `[p, q, limit]`, whether a path joins p and q using only edges lighter than the limit. Answer the queries offline: sort them by limit, let the edges in by weight, and each query becomes one `find` comparison, written into its original slot.
- Sort Items by Groups Respecting Dependencies (1203) asks for an order of items that respects every "before" rule and keeps each group's items together. Give every ungrouped item its own new group, topo-sort the items and, separately, the groups (`group[u] -> group[v]` when they differ), then output the groups in group order with each group's items in item order; either sort failing means `[]`.
- Redundant Connection II (685) adds one extra *directed* edge to a rooted tree and asks which edge to remove. If some node now has two parents, the answer is one of those two edges: skip the later one and union the rest; if no cycle appears, the later edge is the answer, and otherwise the earlier one is. If no node has two parents, return the first edge that closes a cycle.
- Remove Max Number of Edges to Keep Graph Fully Traversable (1579) asks how many edges can go while Alice and Bob, each with edges only they may use plus shared ones, still reach every node. Keep one DSU per player: shared edges go first and are kept when they merge anything, then each player's own. The answer is edges − kept, or −1 if either player's DSU still has more than one group.
- Minimize Malware Spread (924) asks which one node to take off the initially infected list so that the fewest nodes end up infected. Union the adjacency matrix, count the infected nodes per root, and remove the infected node whose component holds exactly one infected node and is the largest; ties go to the smallest index, and with no such node the answer is `min(initial)`.

### Say it in the interview

> "Prerequisites form a directed graph, and I need an order that respects every edge: a topological sort. The brute force rescans all edges for a course with nothing pending, O(V · (V + E)). I'll keep an indegree counter per course and a queue of courses at zero; taking a course only decrements its successors, so every edge is touched once, O(V + E). If fewer than n courses come out, the rest are stuck on a cycle."

> "Groups only ever merge, so I'll use union-find with path halving and union by size: amortised α(n) per operation, effectively constant. A union that finds both ends already under the same root is exactly the edge that closes a cycle."

Point at the `indegree[v] == 0` line, the `len(order) == n` check, and the `if ra == rb: return False` line: those three lines are the whole idea. Follow-ups to expect: *the smallest order?* (a heap instead of the deque), *is the order unique?* (no, if the queue ever holds two nodes at once), *show me the cycle* (DFS colours: when you meet a GRAY node, the GRAY path from it to the current node is the cycle), and *how fast is union-find?* (α(n) amortised, as above).

### Problem map

| Problem | Where | Key insight |
|---|---|---|
| Accounts Merge | `graphs/accounts_merge.py` | email → first account index; union on every repeat; bucket emails by root |
| Alien Dictionary | `graphs/alien_dictionary.py` | neighbouring words give one edge at their first difference; a word before its prefix → ""; Kahn |
| Checking Existence of Edge Length Limited Paths | `graphs/checking_existence_of_edge_length_limited_paths.py` | offline: sort queries by limit, union the edges lighter than it, answer in the original order |
| Course Schedule | `graphs/course_schedule.py` · `practice/simple/43_course_schedule.py` | Kahn: possible iff all n courses get popped |
| Course Schedule II | `graphs/course_schedule_ii.py` | Kahn's pop order is the answer; shorter than n means a cycle → [] |
| Critical Connections in a Network | `graphs/critical_connections_in_a_network.py` | Tarjan: tree edge (u, v) is a bridge iff low[v] > disc[u] |
| Find All People With Secret | `graphs/find_all_people_with_secret.py` | union one timestamp at a time; reset everyone not connected to person 0 |
| Find the Town Judge | `graphs/find_the_town_judge.py` | score = in-degree − out-degree; only the judge reaches n − 1 |
| Graph Valid Tree | `graphs/graph_valid_tree.py` | exactly n − 1 edges and no union fails |
| Largest Component Size by Common Factor | `graphs/largest_component_size_by_common_factor.py` | union each number with the first owner of each prime factor; count numbers per root |
| Minimize Malware Spread | `graphs/minimize_malware_spread.py` | removing a node saves its component only if it is the only infected one; pick the biggest (ties: smallest index; none: min(initial)) |
| Number of Connected Components in an Undirected Graph | `graphs/number_of_connected_components.py` | start at n; every successful union subtracts one |
| Number of Islands II | `graphs/number_of_islands_ii.py` | new land: +1, then −1 for each distinct neighbouring root it merges |
| Parallel Courses III | `graphs/parallel_courses_iii.py` | longest path in a DAG: push finish times forward in Kahn order |
| Reconstruct Itinerary | `graphs/reconstruct_itinerary.py` | Hierholzer: fly the smallest ticket, write an airport when stuck, reverse |
| Redundant Connection | `graphs/redundant_connection.py` · `practice/simple/47_redundant_connection.py` | the first union whose two ends already share a root |
| Redundant Connection II | `graphs/redundant_connection_ii.py` | a two-parent node? try dropping the later edge, else the earlier; otherwise the cycle edge |
| Remove Max Number of Edges to Keep Graph Fully Traversable | `graphs/remove_max_number_of_edges_to_keep_graph_fully_traversable.py` | two DSUs; shared edges first; removable = edges − kept |
| Sort Items by Groups Respecting Dependencies | `graphs/sort_items_by_groups_respecting_dependencies.py` | topo-sort items and groups separately; loners get their own group |

### Self-check

1. You reversed every edge in Course Schedule by mistake. Why does part I still pass while part II fails?
<details><summary>Answer</summary>Reversing every edge turns each cycle into a cycle and each acyclic graph into an acyclic graph, so "is there an order?" has the same answer. But the order you return is valid for the <em>reversed</em> graph: every course now comes out before its own prerequisites, so part II's answer is wrong.</details>

2. Kahn's algorithm stopped with `len(order) < n`. What can you say about the nodes left over?
<details><summary>Answer</summary>Each of them still has indegree at least 1, and every unfinished predecessor of a leftover node is itself a leftover. Walking backwards along those edges never has to stop, so in a finite graph it must repeat a node: there is a cycle among the leftovers. (Some leftovers may only sit downstream of the cycle.)</details>

3. What does `union(a, b)` returning False tell you, and which problems use exactly that?
<details><summary>Answer</summary>a and b were already connected before this edge, so the edge closes a cycle. Redundant Connection returns that edge, Graph Valid Tree fails on it, a component count ignores it, and Kruskal's MST (<a href="#s19">Graphs III</a>) skips it.</details>

4. When does union-find beat a BFS for "are a and b connected?"
<details><summary>Answer</summary>For a fixed graph and a single question, one BFS or DFS is simplest and just as fast. When edges arrive over time, when there are many questions between the arrivals, or when you need "the edge that closes a cycle", union-find wins: each operation is nearly O(1) and nothing is recomputed.</details>

## Graphs III: Weighted Paths & MST

> Once edges have prices, "fewest edges" is no longer "cheapest". **Dijkstra** is BFS with a priority queue: always settle the closest unsettled node, because with no negative edges nobody can reach it more cheaply later. **A minimum spanning tree** connects everything for the least total price: keep taking the cheapest edge that does not close a loop.

[Graphs I](00_Topic_Index.ipynb#s17) counted every move as one step, so the first arrival at a node was the cheapest. Once edges carry prices that stops being true: a heap takes over from the queue, and the union-find of [Graphs II](00_Topic_Index.ipynb#s18) comes back to build spanning trees.

**Reach for it when** edges have weights and the problem asks for the minimum cost, time or effort to get somewhere: that is Dijkstra. When every weight is 0 or 1, a free move or a paid move, a deque replaces the heap, the 0-1 BFS. When a route is only as hard as its *worst* step, Dijkstra runs with `max` in place of `+`.

"Connect all points or cities as cheaply as possible" is a minimum spanning tree, built by Prim or Kruskal, and "at most k stops" or negative weights call for Bellman-Ford rounds.

**In this repo:** `graphs/` (7 of its 45 problems) · bank: `practice/simple/45_network_delay_time.py`, `practice/simple/46_min_cost_to_connect_all_points.py` · basics: `practice/simple/basics/graphs/06_dijkstra.py`, `practice/simple/basics/graphs/05_prim_mst.py`, `practice/simple/basics/graphs/04_kruskal_mst.py`, `practice/simple/basics/graphs/03_union_find.py`

### The picture

```text
         1
   A ---------- B          pop (0,A)  settle A  push B=1 C=5   heap (1,B) (5,C)
   |            |          pop (1,B)  settle B  push D=3       heap (3,D) (5,C)
 5 |            | 2        pop (3,D)  settle D  push C=4       heap (4,C) (5,C)   C got cheaper
   |            |          pop (4,C)  settle C                 heap (5,C)
   C ---------- D          pop (5,C)  stale: 5 > dist[C] = 4, skip it
         1
```

The heap always hands out the closest node nobody has settled yet. C was first priced at 5, straight from A, then found again at 4 along A, B, D, C; the old entry is not removed but skipped when it surfaces. That skip is the **stale check** (lazy deletion, as in [Heaps](15_Heaps.ipynb#topic-heaps)).

Why is a popped node final?

```text
   settled (popped)                    in the heap (priced, not final yet)
( src ... x ) --w-->  y  [price(y) >= d]  ... -->  u

u is popped with d = the smallest price in the heap. Any other route to u leaves the
settled blob through some edge x -> y; at y it already costs >= price(y) >= d, and the
rest of the route adds >= 0 (no negative edges). Nothing can beat d.
```

```text
    0 ---4--- 1                 Kruskal: cheapest edge first; skip it if its ends are already connected
    |       / |
    1     2   5                 0-2 (1)  take   {0,2} {1} {3} {4}
    |   /     |                 1-2 (2)  take   {0,1,2} {3} {4}
    2 ---8--- 3 ---3--- 4       3-4 (3)  take   {0,1,2} {3,4}
                                0-1 (4)  SKIP   0 and 1 are already connected: it would close a loop
                                1-3 (5)  take   {0,1,2,3,4}: n - 1 = 4 edges, done, total 11

    Prim grows one blob from 0 instead, always adding the cheapest edge that leaves it:
    0-2 (1), 2-1 (2), 1-3 (5), 3-4 (3): the same total, 11
```

**Why the greedy edge is safe.** The argument is the cut property. Split the nodes into two sides and let e be the cheapest edge crossing the split. Take any best tree that does not use e: adding e closes a loop, and that loop must cross the split a second time through some edge f with w(f) ≥ w(e). Swap f for e: the tree still spans everything and is never heavier, so some best tree uses e.

Prim uses the split "tree so far | the rest", so the cheapest edge leaving its tree is always safe to add.

Kruskal, at an edge (u, v) whose ends have different roots, uses the split "u's group | the rest". Every cheaper edge has already been processed, and none of them can still leave u's group, because it would have been taken; so (u, v) is the cheapest edge leaving that group, and safe.

**Why it is fast.** To relax an edge u → v is to ask whether going through u makes v cheaper, and to lower v's price if it does. Bellman-Ford relaxes every edge in V − 1 rounds, O(V · E), because it does not know which distances are final. Dijkstra settles nodes in order of distance, so each node's edges are relaxed once, and the heap hands out the next node in O(log V): O((V + E) log V).

For the MST, the brute force tries spanning trees, and there are exponentially many. The cut property makes the greedy choice safe, so one sorted pass (Kruskal) or one heap-driven growth (Prim) is enough.

### From idea to code

**The idea in one sentence:** *pop the cheapest entry from a min-heap; if it is out of date, skip it; otherwise it is final, so offer its neighbours at their new prices.* Dijkstra prices a node by the whole path `d + w`; Prim prices it by the single edge `w` that would attach it to the tree. That is the one real difference between the two loops: both pop the cheapest entry, skip the out-of-date ones, and offer the neighbours.

For Dijkstra the **state** is `dist` and a min-heap of `(dist, node)` entries. By **definition**, `dist[v]` is the cheapest path from `src` to v found so far, and it is final once v is popped with `d == dist[v]`. The **invariant**: nodes are settled in nondecreasing distance, and a settled distance never changes. A **step** pops `(d, u)`, skips it if `d > dist[u]`, and relaxes each edge `(v, w)`: if `d + w < dist[v]`, it lowers `dist[v]` and pushes the new price.

The **record** is `dist[u]`, final at its non-stale pop. **Init** is `dist = {src: 0}` and the heap `[(0, src)]`. The **return** is `dist`, `dist[target]` or `max(dist.values())`, and −1 when a node never appears.

Prim makes the same decisions about a tree. Its state is `in_tree` and a min-heap of `(edge cost, node)`, where an entry `(w, v)` means "v could join the tree through an edge of cost w". Its invariant is that the tree built so far is part of some minimum spanning tree. A step pops `(w, v)`, skips it if v is already in the tree, adds v, and pushes `(w', x)` for every edge to an outside node x.

Prim records `total += w` when v joins. Init is the heap `[(0, start)]`, so the start joins for free, and the return is `total` once all n have joined; fewer means the graph is disconnected.

The sentences map onto lines as usual. "The closest node not settled yet" is `d, u = heapq.heappop(heap)`, and "an out-of-date copy of u" is `if d > dist[u]: continue`. "Going through u is cheaper" is `if d + w < dist.get(v, math.inf):`, and "remember it and offer it" is `dist[v] = d + w` followed by a push.

Network Delay Time sends a signal from node k along directed edges, each with a travel time, and asks when the last node hears it, or -1 if some node never does: with the edges 2 → 1, 2 → 3 and 3 → 4, each taking 1, a signal from 2 needs 2. So `dijkstra` computes every shortest distance and the wrapper takes the largest. The whole point is the order of two moments: a node is final when it is *popped* past the stale check, never when it is pushed.

### Where it goes wrong

1. **Marking a node when it is pushed (the BFS habit).** A cheaper route can still arrive through a node popped later. With `seen` on push, the second delay example answers 6 instead of 4 (plain BFS gives the same 6).
2. **Stopping when the target is discovered.** Its first price may not be its best: from 1 to 2 in the second delay example you get 4 instead of 2. Return when the target is *popped*.
3. **No stale check while recording at pop time.** Distances still come out right (an out-of-date `d` cannot improve anything), but whatever you record per pop is wrong: the max over pops in the trace graph is 5 instead of 4, and Prim counts a point twice (18 instead of 20).
4. **Negative edges.** "Popped means final" needs `w >= 0`. With `A->B 1, A->C 2, C->B -2, B->D 1`, B is popped at 1 before the cheaper route through C (0) is found. A version that never re-expands a node (a `done` set) leaves D at 2 instead of 1; this lazy version re-pushes B and recovers here, but can take exponential time in general. Negative weights need Bellman-Ford.
5. **0-1 BFS with a plain FIFO.** A free move must go to the *front*; appending it lets a cost-1 entry come out first, and `[[1, 1, 3], [3, 2, 2], [1, 1, 4]]` answers 1 instead of 0.
6. **Bottleneck with the wrong key.** Pushing the step alone instead of `max(t, step)` forgets the worst step so far: Swim in Rising Water, the lowest water level at which you can swim from one corner to the other, answers 6 instead of 16 on the 5 × 5 grid in Variations.
7. **Unreachable nodes.** They never get a `dist`. Check `len(dist) == n` before taking a max, and return −1 otherwise: without the check, `network_delay_time([[1, 2, 1]], 3, 1)` answers 1, although node 3 never hears the signal.
8. **Nodes missing from the graph dict.** `dijkstra({0: [(1, 1)]}, 0)` raises `KeyError: 1`, because node 1 has no entry. Build the graph as a `defaultdict(list)`.
9. **Prim's start with a cost.** Pushing `(5, 0)` instead of `(0, 0)` adds 5 to the total (25 instead of 20). The start joins for free.
10. **Prim on a disconnected edge list.** `while joined < n` pops from an empty heap once the reachable part is used up: three nodes with the single edge 0-1 raise `IndexError` after two joins. Loop `while heap and joined < n`, and return −1 if `joined < n`.
11. **Losing edge indexes.** Find Critical and Pseudo-Critical Edges in Minimum Spanning Tree must name edges by their input index. Sorting `[[0, 1, 2], [1, 2, 1]]` in place moves the weight-1 edge to index 0, so it is reported as edge 0 when the input called it edge 1. Sort a list of indexes.
12. **Unorderable heap entries.** On a tie in distance, Python compares the next item of the tuple: pushing `(1, obj)` twice with plain objects raises `TypeError`. Push `(dist, index)` or `(dist, counter, obj)`, as in [Heaps](15_Heaps.ipynb#topic-heaps).

### Edge cases to say out loud

The source alone · an unreachable node (−1) · zero-weight edges · parallel edges (the cheaper one wins) · self-loops (harmless) · one point (MST cost 0) · duplicate points (cost 0) · a disconnected graph for an MST (fewer than n − 1 edges) · equal weights (several MSTs, one total).

### Variations

Every variation keeps the pop, skip and relax loop and changes one thing: the container, the key, the number of rounds, or what a node is.

| Variation | What changes from the template | Problems |
|---|---|---|
| **All distances, then a max** | run to the end; answer = `max(dist.values())`, −1 if a node is missing | Network Delay Time (743) |
| **Weights are only 0 or 1** | a deque instead of a heap: weight 0 → `appendleft`, weight 1 → `append` | Minimum Cost to Make at Least One Valid Path in a Grid (1368), Minimum Moves to Move a Box to Their Target Location (1263) |
| **Dijkstra on a grid or on states** | a node is `(r, c)` or a whole state like `(box, player)`; the loop is unchanged | Path With Minimum Effort (1631), Swim in Rising Water (778), Minimum Moves to Move a Box to Their Target Location (1263) |
| **Minimise the worst step** | key = `max(so_far, step)` instead of a sum | Path With Minimum Effort (1631), Swim in Rising Water (778) |
| **Most likely path** | push `-p`: a product of probabilities ≤ 1 never grows along a path | Path with Maximum Probability (1514) |
| **At most k edges, or negative weights** | Bellman-Ford: k + 1 rounds, each relaxing every edge from a copy of last round's prices | Cheapest Flights Within K Stops (787) |
| **MST on a set of points** | Prim with a heap, or the O(n²) array version on a complete graph | Min Cost to Connect All Points (1584) |
| **MST from an edge list** | Kruskal: sort + union-find | Min Cost to Connect All Points (1584), with every pair listed as an edge |
| *Second pass:* **several fixed endpoints** | Dijkstra from each source, and from the target on reversed edges; minimise over the meeting node | Minimum Weighted Subgraph With the Required Paths (2203) |
| *Second pass:* **which edges matter to the MST** | rerun Kruskal once per edge, skipping it or forcing it in first | Find Critical and Pseudo-Critical Edges in Minimum Spanning Tree (1489) |

Two weight patterns come first, and the simpler one lets a deque replace the heap. When every weight is 0 or 1, the heap only ever holds two distances, `d` and `d + 1`, and a deque keeps that order for free: a free move joins the *front* of the line with `dq.appendleft(...)`, a paid move the back with `dq.append(...)`.

Minimum Cost to Make at Least One Valid Path in a Grid puts an arrow in every cell: right, left, down or up. Following a cell's arrow is free, turning it to point elsewhere costs 1, and the problem asks for the cheapest way to make a path from the top-left to the bottom-right corner: 3 for the first grid below.

Find Critical and Pseudo-Critical Edges in Minimum Spanning Tree asks which edges every minimum spanning tree uses, the critical ones, and which only some of them use, the pseudo-critical ones. Kruskal is cheap, so run it once per question about one edge.

An edge is critical when skipping it makes the tree heavier, or impossible, where a run that cannot span counts as infinitely heavy. An edge that is not critical is pseudo-critical when forcing it in first still gives the best weight. Sort a list of indexes, not the edges themselves, so that the answer keeps the input's numbering (trap 11).

### Say it in the interview

> "The edges have different costs, so 'fewest edges' isn't 'cheapest' and plain BFS is out. Bellman-Ford would relax every edge V − 1 times, O(V · E). The weights are non-negative, so I'll use Dijkstra: a min-heap of (distance, node)."

> "When I pop a node with distance d, any other route to it must leave the settled nodes through something still in the heap, which already costs at least d, and edges can't subtract, so d is final and I relax its edges once. Entries that went out of date are skipped when they surface. With lazy deletion the heap holds up to E entries: O(E log E) = O(E log V)."

> "We need every point connected at minimum total cost, which is a minimum spanning tree. The cheapest edge leaving the tree built so far is always safe, so Prim's grows the tree with a heap of (edge cost, point): O(n² log n) on the complete graph of n points, or O(n²) with the array version."

Point at the stale check, the relax condition, and the line where the answer is read. If the weights are only 0 and 1, say "0-1 BFS" and show the two `append` calls. Follow-ups to expect: negative weights (Bellman-Ford), at most k stops (rounds with a copy), the path itself (`parent[v] = u` on every successful relax, then walk back), a single target (return at its first non-stale pop), and a complete graph (the O(V²) array version).

### Problem map

| Problem | Where | Key insight |
|---|---|---|
| Find Critical and Pseudo-Critical Edges in Minimum Spanning Tree | `graphs/find_critical_and_pseudo_critical_edges_in_minimum_spanning_tree.py` | critical: the MST without it is heavier; pseudo: forcing it in first keeps the MST weight |
| Min Cost to Connect All Points | `graphs/min_cost_to_connect_all_points.py` · `practice/simple/46_min_cost_to_connect_all_points.py` | Prim on the complete graph: heap of (distance, point), skip points already in; or O(n²) with an array |
| Minimum Cost to Make at Least One Valid Path in a Grid | `graphs/minimum_cost_to_make_at_least_one_valid_path_in_a_grid.py` | 0-1 BFS: following the arrow costs 0 (front), turning it costs 1 (back) |
| Minimum Moves to Move a Box to Their Target Location | `graphs/minimum_moves_to_move_a_box_to_their_target_location.py` | 0-1 BFS over (box, player) states: walking costs 0, a push costs 1 |
| Minimum Weighted Subgraph With the Required Paths | `graphs/minimum_weighted_subgraph_with_the_required_paths.py` | Dijkstra from src1, src2, and from dest on reversed edges; minimise over the meeting node |
| Network Delay Time | `graphs/network_delay_time.py` · `practice/simple/45_network_delay_time.py` | Dijkstra from k; answer = the largest distance, −1 if a node is never reached |
| Swim in Rising Water | `graphs/swim_in_rising_water.py` | bottleneck Dijkstra: key = the highest elevation so far; the first pop of the corner wins |

### Self-check

1. Why is a node's distance final when it is popped (and not stale)? What breaks with a negative edge?
<details><summary>Answer</summary>Every other route to that node has to leave the settled region through some node still in the heap, and that node is already at least as far away. Non-negative edges can only add to that, so no other route can be cheaper. A negative edge can subtract, so a route through a node popped <em>later</em> can still undercut a settled distance.</details>

2. In 0-1 BFS, why must a free move go to the front of the deque?
<details><summary>Answer</summary>The deque must stay sorted by cost, like a heap. A free move has the same cost <code>d</code> as the node being expanded, which is the smallest cost in the deque, so it belongs in front of every <code>d + 1</code> entry. Put at the back, it waits behind more expensive entries, and the target can be popped with a cost that is too high.</details>

3. Prim and Dijkstra look almost identical. What is the one real difference?
<details><summary>Answer</summary>The key in the heap. Dijkstra orders nodes by the whole path cost <code>d + w</code> from the source; Prim orders them by the single edge <code>w</code> that would attach them to the tree.</details>

4. Why can't Dijkstra mark a node when it is pushed, as BFS does?
<details><summary>Answer</summary>In BFS every edge costs 1, so the first discovery comes from the closest ring and is already final. With weights, a node first discovered through an expensive edge can still be reached more cheaply through a node that is popped later. Marking it at push time blocks that cheaper route: the second delay example then answers 6 instead of 4.</details>